# Spatial Transcriptomics Workflow Part 3

In [2]:
# Install scanpy, squidpy and matplotlib
!pip install -q scanpy squidpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
# Import libraries
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [4]:
# Loading prepared AnnData object
adata = sc.read_h5ad("breast_cancer_spatial_clustered_complete.h5ad")

In [5]:
adata

AnnData object with n_obs × n_vars = 2516 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'spatial_domains'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'spatial', 'spatial_domains', 'spatial_domains_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## Marker Gene Discovery (Differential Expression)

In [6]:
# IDENTIFY SPATIAL DOMAIN BIOMARKERS
sc.tl.rank_genes_groups(
    adata,
    groupby="spatial_domains",
    method="wilcoxon",       # Non-parametric rank-sum test
    use_raw=True             # Test using un-subsampled raw expression values
)

# Preview of the top 5 genes per cluster
marker_df = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(5)
print("\n📊 Top 5 Marker Genes per Spatial Domain:")
print(marker_df)


📊 Top 5 Marker Genes per Spatial Domain:
  Luminal/Secretory Epithelial Activated Fibroblast/Myofibroblast  \
0                          LTF                             COL4A1   
1                          CLU                              TIMP3   
2                         FTH1                               CST1   
3                       ARPC1B                              MMP11   
4                       PABPC1                                FN1   

  Proliferating Tumor Core Myeloid/Macrophage Infiltrate  \
0                     ASPH                          PSAP   
1                    PSMD3                          CTSB   
2                  PPP1R1A                         HMOX1   
3                     FTH1                           FTL   
4                   CRABP2                          CTSD   

  Dense Collagenous Stroma T-Cell/Lymphoid Zone Luminal epithelial cells  \
0                   COL3A1                TRBC2                    AZGP1   
1                   COL1A2    

## Spatially Variable Genes Identification

In [7]:
# ==============================================================================
# 1. BUILD THE PHYSICAL NEIGHBOR GRAPH
# ==============================================================================
print("Building spatial neighborhood graph network")
sq.gr.spatial_neighbors(adata)
# This automatically adds 'spatial_connectivities' to adata.obsp behind the scenes

# ==============================================================================
# 2. CALCULATE MORAN'S I SPATIAL AUTOCORRELATION
# ==============================================================================
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=adata.var_names[:100]  # Calculates for the first 100 genes (or pass specific genes)
)

# 3. View the top spatially variable genes
print("\nTop spatially clustered genes:")
print(adata.uns["moranI"].head(10))

Building spatial neighborhood graph network
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           

Top spatially clustered genes:
                 I  pval_norm  var_norm  pval_norm_fdr_bh
ISG15     0.597816        0.0  0.000146               0.0
TACSTD2   0.590288        0.0  0.000146               0.0
C1QB      0.552057        0.0  0.000146               0.0
KIAA1324  0.545945        0.0  0.000146               0.0
LAPTM5    0.543199        0.0  0.000146               0.0
PDZK1IP1  0.509859        0.0  0.000146               0.0
SERINC2   0.501783        0.0  0.000146               0.0
DHCR24    0.487416        0.0  0.000146               0.0
PTPRF     0.451686        0.0  0.000146               0.0
NBL1      0.436871        0.0  0.000146               0.0


In [8]:
# 1. Grab the dataframe
svg_df = adata.uns["moranI"].copy()

# 2. FILTERING
significant_svgs = svg_df[
    (svg_df["I"] > 0.2) &                  # Select moderately-to-highly clustered genes
    (svg_df["pval_norm_fdr_bh"] < 0.05)    # Use the FDR-adjusted p-value column
].sort_values(by="I", ascending=False)

# 3. Print out your top 10 Spatially Variable Genes
print(f"🧬 Identified {len(significant_svgs)} significantly spatially variable genes!\n")
print(significant_svgs[["I", "pval_norm_fdr_bh"]].head(10))

# 4. Save to a CSV file
significant_svgs.to_csv("breast_tumor_spatially_variable_genes.csv")
print("\n💾 Full list saved to workspace as 'breast_tumor_spatially_variable_genes.csv'")

🧬 Identified 25 significantly spatially variable genes!

                 I  pval_norm_fdr_bh
ISG15     0.597816               0.0
TACSTD2   0.590288               0.0
C1QB      0.552057               0.0
KIAA1324  0.545945               0.0
LAPTM5    0.543199               0.0
PDZK1IP1  0.509859               0.0
SERINC2   0.501783               0.0
DHCR24    0.487416               0.0
PTPRF     0.451686               0.0
NBL1      0.436871               0.0

💾 Full list saved to workspace as 'breast_tumor_spatially_variable_genes.csv'


###  Spatial Compartmentalization & Tissue Architecture
By overlaying your 9 clustering domains onto the high-resolution tissue histology image (squidpy_breast_tumor_architecture.png), shifted the data from arbitrary mathematical clusters to true anatomical structures.

- The Discovery: The transcriptomic data independently reconstructed the histological layers of Ductal Carcinoma in Situ (DCIS).

- Key Insight: The algorithm clearly demarcated the DCIS Tumor Core from the surrounding Invasive Front, demonstrating that spatial transcriptomics can capture subtle microenvironmental shifts (like the early breakdown of the basement membrane) where tumor cells begin escaping into the stroma.

### Spatially Variable Gene (SVG) Identification
 Moran's $I$ spatial autocorrelation analysis filtered out background transcriptomic noise to highlight the exact genes whose expression is strictly dictated by tissue geography.
 - The Discovery: pulled out highly localized, statistically sound biomarkers ($P_{FDR} < 0.05$) with severe spatial clustering patterns, led by three top candidates:
 - TACSTD2 (Trop-2): A classic epithelial/tumor marker. Its high Moran's $I$ score indicates it is concentrated perfectly within the malignant tumor nests.- ISG15 (Interferon-Stimulated Gene 15): An immune-responsive gene. Its localized clustering flags an active interferon-driven immune response localized to a specific sub-pocket of the tissue.
 - C1QB (Complement Component 1q): Associated with tumor-associated macrophages. Its spatial patterning marks the precise spots where the innate immune system is attempting to infiltrate the tumor margin.

### Summary:
By executing an end-to-end spatial transcriptomics workflow, this project successfully resolved the cellular and architectural landscape of breast cancer tissue at single-spot resolution. Beyond simple spatial clustering, differential expression modeling successfully mapped distinct functional microenvironments.

The pipeline distinctly isolated the metabolic engine of the disease (Proliferating Tumor Core: ASPH, PSMD3), localized critical spatial immune aggregates (Lymphoid Zone: CCL19, CCL5), and successfully identified a clinically critical population of immunosuppressive SPP1+ Macrophages (SPP1, CTSD) operating at the border of an actively degrading extracellular matrix (Remodeling Myofibroblasts: MMP11, MMP2).

This architecture proves that spatial transcriptomics can seamlessly bridge the gap between traditional histology and high-throughput molecular oncology, providing a blueprint for downstream therapeutic target discovery.